In [ ]:
import json
from IPython.display import HTML, display

APP = json.loads('"{\\"title\\": \\"NGO API Triage\\", \\"audience\\": \\"NGO engineers, civic-tech teams, and partner product teams integrating DueCare into an existing stack.\\", \\"decision\\": \\"Choose this when the deployment target is software-to-software integration, not a human-only notebook or phone experience.\\", \\"flow_steps\\": [\\"Receive structured input from a partner system.\\", \\"Run DueCare analysis and scoring.\\", \\"Return action, evidence, and resource guidance.\\", \\"Hand the result back to the caller for triage or case routing.\\"], \\"proof_anchor\\": \\"620 Demo API Endpoint Tour\\"}"')

palette = {
    'primary': '#4c78a8',
    'success': '#10b981',
    'warning': '#f59e0b',
    'info': '#3b82f6',
    'muted': '#6b7280',
    'bg_primary': '#eff6ff',
    'bg_success': '#ecfdf5',
    'bg_warning': '#fffbeb',
    'bg_info': '#eff6ff',
}

def card(label, value, kind='primary'):
    color = palette[kind]
    bg = palette[f'bg_{kind}']
    return (
        f'<div style="display:inline-block;vertical-align:top;width:23%;margin:4px 1%;padding:14px 16px;'
        f'background:{bg};border-left:5px solid {color};border-radius:4px;'
        f'font-family:system-ui,-apple-system,sans-serif">'
        f'<div style="font-size:11px;font-weight:600;color:{color};text-transform:uppercase;letter-spacing:0.04em">{label}</div>'
        f'<div style="font-size:15px;font-weight:600;color:#1f2937;margin-top:6px;line-height:1.35">{value}</div>'
        '</div>'
    )

cards = [
    card('Application', APP['title'], 'primary'),
    card('Primary user', APP['audience'], 'info'),
    card('Use this when', APP['decision'], 'warning'),
    card('Proof anchor', APP['proof_anchor'], 'success'),
]
display(HTML('<div style="margin:10px 0">' + ''.join(cards) + '</div>'))

steps = []
for index, step in enumerate(APP['flow_steps'], start=1):
    steps.append(
        '<div style="display:inline-block;vertical-align:middle;min-width:150px;margin:4px 6px 4px 0;'
        'padding:10px 12px;background:#f8fafc;border:1px solid #d1d5db;border-radius:6px;'
        'font-family:system-ui,-apple-system,sans-serif">'
        f'<div style="font-size:11px;color:#4c78a8;font-weight:600;text-transform:uppercase">Step {index}</div>'
        f'<div style="font-size:13px;color:#1f2937;margin-top:4px">{step}</div>'
        '</div>'
    )

display(HTML(
    '<div style="margin:12px 0 6px 0;font-family:system-ui,-apple-system,sans-serif;font-weight:600;color:#1f2937">'
    'At a glance flow</div>' + ''.join(steps)
))


In [ ]:
# Install the pinned DueCare packages for this notebook.
import glob
import subprocess
import sys

PACKAGES = ['duecare-llm-core==0.1.0']
DUECARE_PACKAGES = [package for package in PACKAGES if package.startswith('duecare-')]
EXTRA_PACKAGES = [package for package in PACKAGES if not package.startswith('duecare-')]

def _pip_install(items):
    if items:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *items])

def _wheel_install():
    wheel_patterns = [
        '/kaggle/input/duecare-llm-wheels/*.whl',
        '/kaggle/input/datasets/taylorsamarel/duecare-llm-wheels/*.whl',
        '/kaggle/input/**/*.whl',
    ]
    wheel_files = []
    for pattern in wheel_patterns:
        wheel_files = sorted(glob.glob(pattern, recursive=True))
        if wheel_files:
            break
    if not wheel_files:
        return False
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', *wheel_files])
    if EXTRA_PACKAGES:
        _pip_install(EXTRA_PACKAGES)
    print(f'Installed {len(wheel_files)} wheel files via attached Kaggle dataset.')
    return True

def _duecare_importable():
    try:
        import importlib
        for mod in ('duecare.core',):
            importlib.invalidate_caches()
            importlib.import_module(mod)
        return True
    except Exception:
        return False

install_source = 'pypi'
try:
    _pip_install(PACKAGES)
except Exception as exc:
    print(f'Pinned PyPI install failed ({exc.__class__.__name__}). Falling back to Kaggle wheels for DueCare packages.')
    if not _wheel_install() and DUECARE_PACKAGES:
        raise RuntimeError('Could not install pinned DueCare packages from PyPI or attached wheel datasets.') from exc
    install_source = 'kaggle_wheels'
else:
    # PyPI pip install returned 0 but that does not guarantee `duecare` is
    # importable (a stub package with the same name can satisfy pip while
    # providing none of the real modules). Verify; fall back to wheels if
    # the import is still broken.
    if DUECARE_PACKAGES and not _duecare_importable():
        print('PyPI install finished but duecare.core is not importable. Falling back to Kaggle wheels.')
        if _wheel_install():
            install_source = 'kaggle_wheels'
        else:
            raise RuntimeError('duecare.core not importable after pip and wheel install. Attach taylorsamarel/duecare-llm-wheels and rerun.')

import duecare.core
print(f'DueCare version: {duecare.core.__version__} ({install_source})')
if duecare.core.__version__ != '0.1.0':
    print('Expected DueCare version: 0.1.0')


# 680: DueCare NGO API Triage

**Expose DueCare as a structured analysis and triage service that other NGO software can call.**

DueCare is an on-device Gemma 4 safety system. This notebook is not the full implementation deep dive. It is the plain-English deployment playbook for one specific application so a judge or adopter can answer a simple question fast: what does this product do, who uses it, and which existing notebooks prove it is real?

<table style="width: 100%; border-collapse: collapse; margin: 4px 0 8px 0;">
  <thead>
    <tr style="background: #f6f8fa; border-bottom: 2px solid #d1d5da;">
      <th style="padding: 6px 10px; text-align: left; width: 22%;">Field</th>
      <th style="padding: 6px 10px; text-align: left; width: 78%;">Value</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="padding: 6px 10px;"><b>Inputs</b></td><td style="padding: 6px 10px;">A JSON request from another application, typically one message, one job post, or one extracted document payload plus optional jurisdiction or context labels.</td></tr>
    <tr><td style="padding: 6px 10px;"><b>Outputs</b></td><td style="padding: 6px 10px;">Structured grades, actions, indicators, legal references, resources, and rubric results that a partner system can store, route, or display downstream.</td></tr>
    <tr><td style="padding: 6px 10px;"><b>Prerequisites</b></td><td style="padding: 6px 10px;">CPU-only narrative notebook. Technical contract lives in <a href="https://www.kaggle.com/code/taylorsamarel/620-duecare-demo-api-endpoint-tour">620 Demo API Endpoint Tour</a>; measured proof lives in <a href="https://www.kaggle.com/code/taylorsamarel/600-duecare-results-dashboard">600 Results Dashboard</a>.</td></tr>
    <tr><td style="padding: 6px 10px;"><b>Runtime</b></td><td style="padding: 6px 10px;">Under 10 seconds. Static request and response rendering only.</td></tr>
    <tr><td style="padding: 6px 10px;"><b>Pipeline position</b></td><td style="padding: 6px 10px;">Deployment Applications section. Previous: <a href="https://www.kaggle.com/code/taylorsamarel/duecare-670-private-client-side-checker">670 Private Client-Side Checker</a>. Next: <a href="https://www.kaggle.com/code/taylorsamarel/duecare-690-migration-case-workflow">690 Migration Case Workflow</a>. Implementation companion: <a href="https://www.kaggle.com/code/taylorsamarel/620-duecare-demo-api-endpoint-tour">620 Demo API Endpoint Tour</a>.</td></tr>
  </tbody>
</table>


### Why this notebook matters

This is the integrator story. A partner deciding whether to wire DueCare into intake tooling, a hotline dashboard, or a monitoring system needs a plain-English API playbook before they need all 17 routes.

### Reading order

- **Previous notebook:** [670 670 Private Client-Side Checker](https://www.kaggle.com/code/taylorsamarel/duecare-670-private-client-side-checker)
- **Next notebook:** [690 690 Migration Case Workflow](https://www.kaggle.com/code/taylorsamarel/duecare-690-migration-case-workflow)
- **Measured proof:** [600 Results Dashboard](https://www.kaggle.com/code/taylorsamarel/600-duecare-results-dashboard)
- **Implementation detail:** [620 Demo API Endpoint Tour](https://www.kaggle.com/code/taylorsamarel/620-duecare-demo-api-endpoint-tour)
- **Scoring detail:** [410 LLM Judge Grading](https://www.kaggle.com/code/taylorsamarel/duecare-410-llm-judge-grading)
- **Section close:** [899 Solution Surfaces Conclusion](https://www.kaggle.com/code/taylorsamarel/duecare-solution-surfaces-conclusion)
- **Back to navigation:** [000 Index](https://www.kaggle.com/code/taylorsamarel/duecare-000-index)

### What this notebook does

1. Defines the NGO API in deployment terms: caller, payload, response, and handoff.
2. Shows one structured request and one structured response with the fields a partner system would consume.
3. Separates the generalized API story from the heavier migration-case orchestration path in 690.


## At a glance

The cards below define the application, the primary user, the decision rule for when to deploy it, and the basic operational flow.

## Composite example

This is the simplest believable input and output shape for the application. It is not meant to replace the implementation notebook; it exists so the product story is visible at a glance.

In [ ]:
import json
from IPython.display import HTML, display

PAYLOAD = json.loads('"{\\"sample_input\\": {\\"text\\": \\"Agency says the worker must reimburse visa, medical, and training fees after arrival.\\", \\"context\\": \\"job_posting\\", \\"jurisdiction\\": \\"PH_HK\\", \\"caller\\": \\"ngo-intake-dashboard\\"}, \\"sample_output\\": {\\"grade\\": \\"bad\\", \\"action\\": \\"review\\", \\"indicators\\": [\\"illegal recruitment fees\\", \\"cost shifting\\"], \\"legal_refs\\": [\\"ILO C181 Article 7\\"], \\"resources\\": [\\"trusted hotline referral\\"], \\"triage_note\\": \\"Send to labor-rights reviewer for manual follow-up.\\"}}"')

input_json = json.dumps(PAYLOAD['sample_input'], indent=2)
output_json = json.dumps(PAYLOAD['sample_output'], indent=2)

display(HTML(
    '<table style="width:100%;border-collapse:collapse;margin:4px 0 8px 0;font-family:system-ui,-apple-system,sans-serif">'
    '<thead><tr style="background:#f6f8fa;border-bottom:2px solid #d1d5da">'
    '<th style="padding:8px 10px;text-align:left;width:50%">Composite input</th>'
    '<th style="padding:8px 10px;text-align:left;width:50%">Expected output shape</th>'
    '</tr></thead><tbody><tr>'
    '<td style="padding:8px 10px;vertical-align:top;border-top:1px solid #e5e7eb"><pre style="white-space:pre-wrap;margin:0">' + input_json + '</pre></td>'
    '<td style="padding:8px 10px;vertical-align:top;border-top:1px solid #e5e7eb"><pre style="white-space:pre-wrap;margin:0">' + output_json + '</pre></td>'
    '</tr></tbody></table>'
))


## Repo evidence

The application notebook is only useful if it points back to the notebooks that make the claim defensible.

In [ ]:
import json
from IPython.display import HTML, display

EVIDENCE = json.loads('"{\\"evidence\\": [{\\"notebook\\": \\"620 Demo API Endpoint Tour\\", \\"url\\": \\"https://www.kaggle.com/code/taylorsamarel/620-duecare-demo-api-endpoint-tour\\", \\"proof\\": \\"Shows the actual request and response shapes that make this application real.\\"}, {\\"notebook\\": \\"600 Results Dashboard\\", \\"url\\": \\"https://www.kaggle.com/code/taylorsamarel/600-duecare-results-dashboard\\", \\"proof\\": \\"Shows the measured quality behind the JSON a partner would trust.\\"}, {\\"notebook\\": \\"410 LLM Judge Grading\\", \\"url\\": \\"https://www.kaggle.com/code/taylorsamarel/duecare-410-llm-judge-grading\\", \\"proof\\": \\"Explains the graded fields and dimensions the response exposes.\\"}, {\\"notebook\\": \\"430 Rubric Evaluation\\", \\"url\\": \\"https://www.kaggle.com/code/taylorsamarel/duecare-430-rubric-evaluation\\", \\"proof\\": \\"Shows the explicit pass-fail criteria behind the triage response.\\"}]}"')['evidence']

rows = []
for item in EVIDENCE:
    rows.append(
        '<tr>'
        f'<td style="padding:8px 10px;vertical-align:top;border-top:1px solid #e5e7eb"><a href="{item["url"]}">{item["notebook"]}</a></td>'
        f'<td style="padding:8px 10px;vertical-align:top;border-top:1px solid #e5e7eb">{item["proof"]}</td>'
        '</tr>'
    )

display(HTML(
    '<table style="width:100%;border-collapse:collapse;margin:4px 0 8px 0;font-family:system-ui,-apple-system,sans-serif">'
    '<thead><tr style="background:#f6f8fa;border-bottom:2px solid #d1d5da">'
    '<th style="padding:8px 10px;text-align:left;width:30%">Supporting notebook</th>'
    '<th style="padding:8px 10px;text-align:left;width:70%">What it proves</th>'
    '</tr></thead><tbody>' + ''.join(rows) + '</tbody></table>'
))


## Boundaries and handoff

- This is the generalized software integration path. It does not assume a multi-document bundle or case timeline.
- Use 690 when the input is a case file with several documents and the output needs to include a chronology and complaint draft.

This notebook is intentionally short. Its job is to make the deployment application legible in minutes, then hand you to the heavier implementation notebook if you want the mechanics.


In [ ]:
print('NGO API Triage handoff >>> 690 690 Migration Case Workflow https://www.kaggle.com/code/taylorsamarel/duecare-690-migration-case-workflow | 600 Results Dashboard https://www.kaggle.com/code/taylorsamarel/600-duecare-results-dashboard | 620 Demo API Endpoint Tour https://www.kaggle.com/code/taylorsamarel/620-duecare-demo-api-endpoint-tour | 650 Custom Domain Walkthrough https://www.kaggle.com/code/taylorsamarel/650-duecare-custom-domain-walkthrough')
